<div style="display: flex; justify-content: space-between;">
<div style="text-align: left; display: inline-block;" align="left"><b>Tepper School of Business </b></div>
<div style="text-align: right; display: inline-block;" align="right"><i>Dennis Epple</i></div>
</div>
<hr>
<div style="display: flex; justify-content: space-between;">
<div style="text-align: left" align="left">Statistics and Decision Making (45-752)</div>
<div style="text-align: right" align="right"><i>2025</i></div>
</div>

In [ ]:
# This command installs tprstats from its GitHub repo. You only need to run this command once, when start the notebook.
#!pip install git+https://github.com/dnepple/tprstats-python@colab

# stress test to see if this is a good solution to problem of installing tprstats
try:
    import tprstats
except ImportError as e:
  !pip install git+https://github.com/dnepple/tprstats-python@colab
  import tprstats

In [ ]:
# ====== Data Setup ======
import os, urllib.request

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

GITHUB_RAW = "https://raw.githubusercontent.com/mhnam/statistical-decision-making/main/data"

# Set working directory so that data/ is directly accessible
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    if os.path.exists("/content/drive/MyDrive/SDM/data"):
        os.chdir("/content/drive/MyDrive/SDM")
    else:
        os.makedirs("data", exist_ok=True)
else:
    for parent in [".", "..", "../.."]:
        if os.path.exists(os.path.join(parent, "data")):
            if parent != ".":
                os.chdir(parent)
            break
    else:
        os.makedirs("data", exist_ok=True)

# Download any missing data files from GitHub
def _download(filename):
    path = f"data/{filename}"
    if not os.path.exists(path):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(f"{GITHUB_RAW}/{filename}", path)

for f in ["Brand_Choice.xlsx", "AWE_Warranty_Experiment.xlsx", "Boston_Condos_572.xlsx"]:
    _download(f)

print(f"Working directory: {os.getcwd()}")

# Assignment 5
This assignment gives you practice with choosing functional form, logit estimation of a model with a (0,1) dependent variable, and analysis of a model with time series data. As with other assignments, this looks long, but I provide guidance along the way so as to use your time efficiently. A separate copy is provided for you without solutions.

## Exercise 1: Logit Estimation and Interpretation
The purpose of this question is to help you gain experience in estimating and applying the results of binary-choice models. We will use logit to investigate the effects of price and promotional activities on purchase of national as compared to private-label brands of ketchup.  Use dataset Brand_Choice.xlsx for these questions.

In [ ]:
import pandas
import tprstats

In [ ]:
Brand_Choice = pandas.read_excel("data/Brand_Choice.xlsx")

Recall the definition of variables:  

The **Dependent variable** is PLBRAND. PLBRAND=1 if the purchase is the private-label brand and PLBRAND=0 if the purchase is a national brand.  

The **Independent variables** measure prices, activities undertaken to market the product, and income in the neighborhood where the product is sold. PRICENAT and PRICEPVT are the dollar prices for a large container of the national and private-label brands respectively.  Variable FEANAT equals one if the national brand was featured in the store’s weekly flyer and zero if it was not.  Similarly, variable FEAPVT equals one if the private-label brand was featured and zero if it was not.  Variable DISPNAT equals one if the national product was displayed at the end of an aisle or in a special display case and zero otherwise.  Variable DISPPVT equals one if the private-label product was displayed at the end of an aisle or in a special display case and zero otherwise. Variable NEIGHINC is the mean income in thousands in the neighborhood where the consumer lives.

### Question 1a(\*)
Estimate a logit model with PLBRAND as the dependent variable. Include the seven variables listed above. Name your model ChoiceLogit. Please report your coefficient of FEANAT.

### Question 1b(\*)
Are all of the variables statistically significant at the 5% level?

### Question 1c(\*)
Do all of the variables have the “right” algebraic signs?

### Question 1d
Calculate the marginal effects of the coefficients. Report the marginal effect of PRICENAT and provide a one-sentence interpretation. In your interpretation, remember to state which other variables, if any, are being held constant.

### Question 1e(\*)
Which exhibits greater price sensitivity, private or national brands?  

### Question 1f(\*)
Do you agree that private-label brands benefit more from being featured in a flyer than national brands?

### Question 1g(\*)
Which benefits more from special display treatment, a national or private-label brand?

### Question 1h
In a large Allmart store, the price of the national brand is \\$4.0 (PRICENAT=4.0), the price of the private-label brand is \\$3.45 (PRICEPVT=3.45), neither the national nor the private brand has a special display (DISPNAT=0, DISPPVT=0), and the national brand is not featured (FEANAT=0). Neighborhood income is \\$35 thousand. You have been asked to calculate the effect of featuring the private-label brand on the probability of sale of the private-label brand. To answer this question, calculate the predicted probability using the values above. Then, do a second prediction with FEAPVT=1. Calculate the difference in probabilities between the two predictions.

### Question 1i(\*)
Suppose Allmart’s marginal profit per sale of the private-label brand is \\$1. This Allmart store typically sells a total of 2,000 bottles of Ketchup per week (the combined total of national and private-label sales). It costs this Allmart store \\$100 to feature the private-label brand for a week. Would it be profitable for this Allmart store to feature the private-label brand if there is no change in the total number of bottles of ketchup sold?  

Hint: In part (h), you calculated the effect of featuring the private-label brand on the probability of purchase. Multiply the change in probability times the number of bottles sold times the marginal profit per sale and deduct the cost of featuring the product.

### Question 1j
Form the logit classification table for your model and report the number of observations for which a sale is predicted to occur and the sale does occur.

## Exercise 2: A logit application combining experimentation, estimation, and optimization
Here is a brief recap of the AWE setting. I provide you with detailed suggestions along the way. This is very similar to the application that we will do in class. Try to do the analysis following these suggestions without looking at the slides for this application.  

Appliance World Enterprises (AWE) owns a chain of retail outlets specializing in the sale and service of home appliances.
- AWE offers extended warranty programs for the appliances it sells.
- A typical refrigerator sold by AWE carries a manufacturer’s warranty that provides full coverage for parts and labor for one year.
- The additional AWE warranty extends this coverage to five years.
- A sale of a five-year warranty is quite profitable for AWE.
- AWE salesmen are trained to encourage appliance purchasers to buy the extended warranty, but only a very small fraction of customers, about 3%, buy the AWE warranty.
- AWE has conducted an experiment to investigate whether reducing the warranty price would increase customer purchase of the warranty.

Details of the experiment:
- Every tenth purchaser of a refrigerator from AWE is invited to choose a small envelope from an urn full of small envelopes.
- When opened, the envelope says “You have won the opportunity to purchase our extended warranty at a discount of __ percent.”
- The percentage discount differs depending on the envelope chosen, and is a value between 10% and 50%.
- The sales person then encourages the customer to purchase the warranty with the discount offered. (A customer can draw only once.)

AWE has obtained results from this experiment for a total of 1,000 customers. The data are in file AWE_Warranty.xlsx. In this data file, X is the discount that an individual received and Y is an indicator equal to 1 if the customer purchased the warranty and 0 if the customer declined.

In [ ]:
AWE_Warranty = pandas.read_excel("data/AWE_Warranty_Experiment.xlsx")

### Question 2a
Estimate a logit model of Y vs X and name the model AWE_logit. Report your result.

### Question 2b
Using the marginal_effects command, calculate and interpret the marginal effect of X.

### Question 2c
Using your estimated model, write the Python command to calculate the predicted probability of purchase if the warranty discount is 33. Report your result.

### Question 2d
Now investigate the optimal warranty for AWE.
- AWE has been pricing its warranty at \\$300.
- If AWE adopts a discount of X%, its price will be 300*(1-X/100).
- Hence, its revenue from each warranty sold will be 300*(1-X/100).
- **AWE has become more efficient in doing repairs. As a result, the present value of the cost of servicing a warranty over five years has fallen from \\$120 to \\$110.**

Write the formula for the expected profit per customer with discount X letting p(X) denote the probability of purchase at discount X.

### Question 2e
Create a range of values of the discount from 0 to 50 in increments of 1 percentage point using the range() command and name the resulting variable discounts.  

Reminder: Python starts counting from 0.

### Question 2f
Using the logit model, calculate predicted purchase probability for each value of discounts. Note that this corresponds to function p(X) above:

### Question 2g
To see the relationship of purchase probability to discount, do a plot with the predicted purchase probabilities on the vertical axis and discounts on the horizontal axis.

### Question 2h
Next, calculate expected profit for each value of discount and name the result profit.

### Question 2i
Plot the result using the command below. From looking at the plot, roughly what percent discount will maximize profit?

### Question 2j
Now, determine more precisely which discount will maximize profit. Here is the command to use: profit.argmax()
Put the command in a code cell and run it.

### Question 2k
Find the discount that maximizes profit. Report the optimal percent discount and the expected profit per warranty at that discount.
Use the following command: profit[41]

## Exercise 3: A logit application to Boston Condominiums
Import Boston_Condos_572.XLSX

In [ ]:
Boston_Condos_572 = pandas.read_excel("data/Boston_Condos_572.xlsx")

### Question 3a
Estimate a logit model to investigate which factors predict that there will be a parking space in the building for a condominium. Your dependent variable is PARK. As independent variables, include FLOORS, I(CDIST/1000), I(SQFT/100). Name your model Parking_logit. Note that the term I(CDIST/1000) will convert distance from feet to thousands of feet. I(SQFT/100) converts square feet to hundreds of square feet.

### Question 3b 

Are all of the coefficients significant at the 5% level?

### Question 3c
Calculate marginal effects from your model. 

### Question 3d
Do you agree with the following. An additonal floor increases the predicted probability of parking in the building by 1.38% holding CDIST and SQFT constant?